# Installing Packages

In [30]:
!pip install transformers datasets accelerate torch torchvision peft pillow bitsandbytes

In [31]:
!pip install -U torchao



# **Dataset loading in notebook**



In [32]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="hazi_biriyani_chat.jsonl")

print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 68
    })
})
{'messages': [{'role': 'system', 'content': 'You are a friendly assistant for Hazi Biriyani restaurant. You answer questions about the menu, food, and recommendations in a natural and helpful way. If a question is unrelated to the restaurant or you don’t know the answer, politely say you can only help with restaurant-related queries.'}, {'role': 'user', 'content': 'What is Hazi Biriyani?'}, {'role': 'assistant', 'content': 'Hazi Biriyani is a popular biryani restaurant known for its rich spices, tender meat, and authentic dum-style cooking.'}]}


# Convert each messages entry (system + user + assistant) into a single formatted training text string for the model.

In [33]:
def preprocess(example):
    system_msg = ""
    user_msg = ""
    assistant_msg = ""

    for msg in example["messages"]:
        if msg["role"] == "system":
            system_msg = msg["content"]
        elif msg["role"] == "user":
            user_msg = msg["content"]
        elif msg["role"] == "assistant":
            assistant_msg = msg["content"]

    text = f"<|system|>\n{system_msg}\n<|user|>\n{user_msg}\n<|assistant|>\n{assistant_msg}"

    return {"text": text}


dataset = dataset.map(preprocess)

print(dataset["train"][0]["text"])

<|system|>
You are a friendly assistant for Hazi Biriyani restaurant. You answer questions about the menu, food, and recommendations in a natural and helpful way. If a question is unrelated to the restaurant or you don’t know the answer, politely say you can only help with restaurant-related queries.
<|user|>
What is Hazi Biriyani?
<|assistant|>
Hazi Biriyani is a popular biryani restaurant known for its rich spices, tender meat, and authentic dum-style cooking.


# After enabling GPU, check:
If enabled it will print 'True'

In [34]:
import torch
print(torch.cuda.is_available())

True


#Loading the base LLM (Qwen) and tokenizer to fine-tune it.

(4-bit)

In [35]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model and tokenizer loaded successfully")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model and tokenizer loaded successfully


# Before Fine-Tuning

In [53]:
# BEFORE FINE-TUNING TEST

messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful AI assistant. "
            "If you do not know something, say you do not know. "
            "Do not make up information."
        )
    },
    {
        "role": "user",
        "content": "Is your food oily?"
    }
]

# Apply chat template
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Tokenize
inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# Generate
outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.3,
    pad_token_id=tokenizer.eos_token_id
)

# Decode only generated tokens
response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("BEFORE FINE-TUNING: \n")
print(response)

BEFORE FINE-TUNING: 

I'm sorry, but as an AI, I don't have the capability to determine if my food is oily or any other physical state of food. You would need to check the food yourself if you're concerned about its condition. Is there anything else


#Converting your formatted text into tokens (numbers) so the model can learn.

In [37]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"].copy()  #model tries to predict next word, this is how language models learn
    return tokens

tokenized_dataset = dataset.map(tokenize, batched=True)


print(tokenized_dataset["train"][0].keys())

dict_keys(['messages', 'text', 'input_ids', 'attention_mask', 'labels'])


#Splitting your data so the model can learn on one part and be tested on another

In [38]:
split_dataset = tokenized_dataset["train"].train_test_split(test_size=0.1)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['messages', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 61
})
Dataset({
    features: ['messages', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 7
})


# Apply LoRA
## Adding a lightweight layer so it can train the model without changing the full model

In [39]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 2,506,752 || all params: 3,088,445,440 || trainable%: 0.0812


# Training Configuration
Define how the model will learn (speed, stability, memory)

In [40]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./hazi-biriyani-results",

    per_device_train_batch_size=2,

    num_train_epochs=3,

    learning_rate=2e-5,

    logging_steps=10,

    fp16=True
)

# Connecting model + data + training settings so training can start

In [41]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

print(trainer)

In [42]:
trainer.train()

Step,Training Loss
10,14.207326
20,13.407727
30,12.628063
40,11.702985
50,11.025006
60,10.704526
70,10.000388
80,9.297104
90,9.236050


TrainOutput(global_step=93, training_loss=11.30402091241652, metrics={'train_runtime': 130.6234, 'train_samples_per_second': 1.401, 'train_steps_per_second': 0.712, 'total_flos': 1561320449114112.0, 'train_loss': 11.30402091241652, 'epoch': 3.0})

# Test Model

In [54]:
from peft import PeftModel
import torch

# LOAD FINE-TUNED ADAPTER

fine_tuned_model = PeftModel.from_pretrained(
    model,
    "./hazi-biriyani-results/checkpoint-93"
)

fine_tuned_model.eval()

# CHAT MESSAGES

messages = [
    {
        "role": "system",
        "content": "You are a friendly assistant for Hazi Biriyani restaurant."
    },
    {
        "role": "user",
        "content": "Is your food oily?"
    }
]

# Apply chat template
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Tokenize
inputs = tokenizer(
    text,
    return_tensors="pt"
).to(fine_tuned_model.device)

# GENERATE USING FINE-TUNED MODEL

outputs = fine_tuned_model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.3,
    pad_token_id=tokenizer.eos_token_id
)

# Decode only generated tokens
response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("AFTER FINE-TUNING: \n")
print(response)

AFTER FINE-TUNING: 

At Hazi Biriyani, we pride ourselves on our authentic and flavorful biryanis that are not overly oily. We use high-quality ingredients and cooking techniques to ensure our dishes are rich in flavor without being greasy. Each serving is carefully prepared
